# Homework 13 - Network Compression

> Author: Arvin Liu (r09922071@ntu.edu.tw), this colab is modified from ML2021-HW3

If you have any questions, feel free to ask: ntu-ml-2021spring-ta@googlegroups.com

## **Intro**

HW13 is about network compression

There are many types of Network/Model Compression, here we introduce two:
* Knowledge Distillation
* Design Architecture

The process of this notebook is as follows:
1. Introduce depthwise, pointwise and group convolution in MobileNet.
2. Design the model of this colab
3. Introduce Knowledge-Distillation
4. Set up TeacherNet and it would be helpful in training

## **About the Dataset** *(same as HW3)*

The dataset used here is food-11, a collection of food images in 11 classes.

For the requirement in the homework, TAs slightly modified the data.
Please DO NOT access the original fully-labeled training data or testing labels.

Also, the modified dataset is for this course only, and any further distribution or commercial use is forbidden.

In [ ]:
# 下載資料集
!gdown --id '1awF7pZ9Dz7X1jn1_QAiKN-_v56veCEKy' --output food-11.zip
!unzip -q food-11.zip

## **Import Packages**

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import ConcatDataset, DataLoader, Subset
from torchvision.datasets import DatasetFolder
from tqdm.auto import tqdm

## **Dataset, Data Loader, and Transforms**

相較 HW3，訓練集的資料增強更積極，加入 ColorJitter 與 RandomGrayscale，幫助模型泛化。

In [ ]:
train_tfm = transforms.Compose([
    transforms.Resize((142, 142)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.RandomCrop(128),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_tfm = transforms.Compose([
    transforms.Resize((142, 142)),
    transforms.CenterCrop(128),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
batch_size = 64

train_set      = DatasetFolder("food-11/training/labeled",   loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
valid_set      = DatasetFolder("food-11/validation",         loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)
unlabeled_set  = DatasetFolder("food-11/training/unlabeled", loader=lambda x: Image.open(x), extensions="jpg", transform=train_tfm)
test_set       = DatasetFolder("food-11/testing",            loader=lambda x: Image.open(x), extensions="jpg", transform=test_tfm)

train_loader = DataLoader(train_set,  batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_set,  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,   batch_size=batch_size, shuffle=False)

## **Architecture — StudentNet**

用 **Depthwise + Pointwise Convolution**（DSC）取代標準 Conv，大幅減少參數量。

### 參數量估算（目標 ≤ 100,000）

| 層 | 說明 | 參數 |
|---|---|---|
| Conv(3→32) | 入口標準 Conv | ~928 |
| DSC(32→64) | Depthwise+Pointwise | ~2,528 |
| DSC(64→128) | Depthwise+Pointwise | ~9,152 |
| DSC(128→128) × 2 | 加深特徵擷取 | ~35,840 |
| FC(128→11) | 分類頭 | ~1,419 |
| **合計** | | **≈ 49,867** |

### DSC 積木說明
```
標準 Conv：  NMkk  個參數
DSC        ：  Nkk + NM  個參數  （N=in_ch, M=out_ch, k=kernel）
節省比例：   NMkk / (Nkk+NM) = Mkk/(kk+M) ≈ k²  倍
```

In [ ]:
def dsc_block(in_chs, out_chs, stride=1):
    """Depthwise Separable Convolution 積木：DW → BN → ReLU → PW → BN → ReLU"""
    return nn.Sequential(
        # Depthwise：每個 channel 各自做 3×3 Conv
        nn.Conv2d(in_chs, in_chs, kernel_size=3, stride=stride, padding=1, groups=in_chs, bias=False),
        nn.BatchNorm2d(in_chs),
        nn.ReLU6(inplace=True),
        # Pointwise：1×1 Conv 混合 channel 資訊
        nn.Conv2d(in_chs, out_chs, kernel_size=1, bias=False),
        nn.BatchNorm2d(out_chs),
        nn.ReLU6(inplace=True),
    )


class StudentNet(nn.Module):
    def __init__(self):
        super(StudentNet, self).__init__()

        self.cnn = nn.Sequential(
            # 入口：標準 Conv 做初步特徵擷取
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=False),  # 128×128×32
            nn.BatchNorm2d(32),
            nn.ReLU6(inplace=True),
            nn.MaxPool2d(2, 2),                                        # 64×64×32

            # DSC Block 1：32 → 64
            dsc_block(32, 64),                                         # 64×64×64
            nn.MaxPool2d(2, 2),                                        # 32×32×64

            # DSC Block 2：64 → 128
            dsc_block(64, 128),                                        # 32×32×128
            nn.MaxPool2d(2, 2),                                        # 16×16×128

            # DSC Block 3 & 4：128 → 128（加深，不擴寬）
            dsc_block(128, 128),                                       # 16×16×128
            dsc_block(128, 128),                                       # 16×16×128

            # Global Average Pooling 代替 Flatten，不受輸入尺寸限制
            nn.AdaptiveAvgPool2d((1, 1)),                              # 1×1×128
        )

        self.fc = nn.Sequential(
            nn.Linear(128, 11),
        )

    def forward(self, x):
        out = self.cnn(x)
        out = out.view(out.size(0), -1)
        return self.fc(out)

## **Model Analysis**

用 `torchsummary` 確認參數量 ≤ 100,000（含 non-trainable）。

In [ ]:
!pip install torchsummary -q
from torchsummary import summary

student_net = StudentNet()
summary(student_net, (3, 128, 128), device="cpu")

## **Knowledge Distillation Loss**

$$Loss = \alpha T^2 \cdot KL\!\left(\frac{\text{Teacher}}{T} \,\|\, \frac{\text{Student}}{T}\right) + (1-\alpha) \cdot CE(\text{Student},\, \text{labels})$$

- **T（Temperature）**：值越大，soft label 分布越平滑，讓 student 學到更多類別間關係
- **α**：soft loss 與 hard loss 的平衡係數
- **T²**：補償 softmax(logits/T) 梯度縮小 T 倍的效果

In [ ]:
def loss_fn_kd(outputs, labels, teacher_outputs, alpha=0.5, T=20):
    """
    Knowledge Distillation Loss
    outputs        : student logits  (batch, num_classes)
    labels         : ground truth    (batch,)
    teacher_outputs: teacher logits  (batch, num_classes)
    alpha          : soft loss 的權重
    T              : temperature，控制 soft label 的平滑程度
    """
    # Hard Loss：student 對 ground truth 的 Cross-Entropy
    hard_loss = F.cross_entropy(outputs, labels) * (1. - alpha)

    # Soft Loss：student 與 teacher soft distribution 的 KL Divergence
    # KLDiv 要求：input 是 log-probability，target 是 probability
    soft_loss = F.kl_div(
        F.log_softmax(outputs / T, dim=1),   # student 的 log soft label
        F.softmax(teacher_outputs / T, dim=1),  # teacher 的 soft label
        reduction='batchmean'
    ) * (alpha * T * T)  # T² 補償梯度縮小

    return hard_loss + soft_loss

## **Teacher Model**

官方提供的 Teacher Net（ResNet，Acc ≈ 0.855）。
只用於產生 soft labels 與 pseudo labels，不更新參數。

In [ ]:
!gdown --id '1zH1x39Y8a0XyOORG7TWzAnFf_YPY8e-m' --output teacher_net.ckpt

teacher_net = torch.load('./teacher_net.ckpt')
teacher_net.eval()

## **Semi-supervised Learning — Pseudo Labels**

用 Teacher Net 對 unlabeled data 預測 pseudo labels，再合併進訓練集。

⚠️ **嚴禁**對 test data 做 pseudo labeling，違規視為作弊。

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

student_net = student_net.to(device)
teacher_net = teacher_net.to(device)

do_semi = True

def get_pseudo_labels(dataset, model):
    """用 model 對 dataset 產生 pseudo labels，直接修改 dataset.samples。"""
    loader = DataLoader(dataset, batch_size=batch_size * 3, shuffle=False, pin_memory=True)
    pseudo_labels = []

    for batch in tqdm(loader):
        img, _ = batch
        with torch.no_grad():
            logits = model(img.to(device))
            pseudo_labels.append(logits.argmax(dim=-1).detach().cpu())

    pseudo_labels = torch.cat(pseudo_labels)

    for idx, ((img_path, _), pseudo_label) in enumerate(zip(dataset.samples, pseudo_labels)):
        dataset.samples[idx] = (img_path, pseudo_label.item())

    return dataset


if do_semi:
    unlabeled_set = get_pseudo_labels(unlabeled_set, teacher_net)
    concat_dataset = ConcatDataset([train_set, unlabeled_set])
    train_loader = DataLoader(concat_dataset, batch_size=batch_size, shuffle=True, pin_memory=True, drop_last=True)

## **Training**

使用 Knowledge Distillation Loss 取代純 Cross-Entropy。

In [ ]:
optimizer = torch.optim.Adam(student_net.parameters(), lr=3e-4, weight_decay=1e-5)

# CosineAnnealingLR：學習率從 3e-4 平滑衰減到接近 0，避免後期震盪
n_epochs = 150
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

best_acc = 0.0

for epoch in range(n_epochs):
    # ---------- Training ----------
    student_net.train()
    train_loss, train_accs = [], []

    for batch in tqdm(train_loader):
        imgs, labels = batch
        logits = student_net(imgs.to(device))

        with torch.no_grad():
            soft_labels = teacher_net(imgs.to(device))

        loss = loss_fn_kd(logits, labels.to(device), soft_labels, alpha=0.5, T=20)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(student_net.parameters(), max_norm=10)
        optimizer.step()

        acc = (logits.argmax(dim=-1) == labels.to(device)).float().mean()
        train_loss.append(loss.item())
        train_accs.append(acc)

    scheduler.step()

    train_loss = sum(train_loss) / len(train_loss)
    train_acc  = sum(train_accs) / len(train_accs)

    # ---------- Validation ----------
    student_net.eval()
    valid_loss, valid_accs = [], []

    for batch in tqdm(valid_loader):
        imgs, labels = batch
        with torch.no_grad():
            logits      = student_net(imgs.to(device))
            soft_labels = teacher_net(imgs.to(device))

        loss = loss_fn_kd(logits, labels.to(device), soft_labels, alpha=0.5, T=20)
        acc  = (logits.argmax(dim=-1) == labels.to(device)).float().detach().cpu().view(-1).numpy()

        valid_loss.append(loss.item())
        valid_accs += list(acc)

    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc  = sum(valid_accs) / len(valid_accs)

    print(f"[ Train | {epoch+1:03d}/{n_epochs:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")
    print(f"[ Valid | {epoch+1:03d}/{n_epochs:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")

    # 儲存 validation acc 最高的 checkpoint
    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(student_net.state_dict(), 'student_best.ckpt')
        print(f"  → Best model saved (acc = {best_acc:.5f})")

## **Testing**

In [ ]:
# 載入最佳 checkpoint
student_net.load_state_dict(torch.load('student_best.ckpt'))
student_net.eval()

predictions = []

for batch in tqdm(test_loader):
    imgs, _ = batch
    with torch.no_grad():
        logits = student_net(imgs.to(device))
    predictions.extend(logits.argmax(dim=-1).cpu().numpy().tolist())

# 儲存預測結果
with open("predict.csv", "w") as f:
    f.write("Id,Category\n")
    for i, pred in enumerate(predictions):
        f.write(f"{i},{pred}\n")

print("Prediction saved to predict.csv")

## **Backup Links**

In [ ]:
# teacher_net 備用連結
# !gdown --id '1VBIeQKH4xRHfToUxuDxtEPsqz0MHvrgd' --output teacher_net.ckpt

# food-11 備用連結
# !gdown --id '1qdyNN0Ek4S5yi-pAqHes1yjj5cNkENCc' --output food-11.zip
# !gdown --id '1c0Q1EP6yIx0O2rqVMIVInIt8wFjLxmRh' --output food-11.zip
# !gdown --id '1hKO054nT1R8egcXY2-tgQbwX4EjowRLz' --output food-11.zip